In [1]:
using Pkg; Pkg.activate(".")
# Pkg.instantiate()

using Dates
using NCDatasets
using PyPlot


  Activating new project at `~/Projects/ASTRAL/soundings/src`


In [18]:
module VaporSat

using ForwardDiff

export qs, dqsdT
export Twet_autodiff

# constants
Cp = 1005.7  # from my Davies-Jones function, was 1005.
Cpv = 1870.0 # J/kg/K
Cw  = 4190.0
L0 = 2.501e6 # J/kg

C = 273.15 # K
Rd = 287.04
Rv = 461.5
RdoRv=Rd/Rv

"latent heat of water vapor"
LvK(TempK) = L0 + (Cpv-Cw) * (TempK-273.0)

# functions
"""
es(T,p) = is saturation vapor pressure based on Wexler's formula,
with enhancement factor for moist air rather than water vapor.
The enhancement factor requires a pressure.
T [degrees C], p [Pa] (note the reversed input order), es [Pa]
Calling with optional keywords changes the units and
ignores the positional arguments.
es(T,p; TK=tk[Kelvin], P=pr[hPa])
From A. L. Buck 1981: JAM, 20, 1527-1532.
SPdeS 7 July 2004
"""
function es(T,p=1e5; P=p*1e-2)
    esat = 1e2 * 6.1121*(1.0007 + 3.46e-8*P)*exp((17.502*T)/(240.97 + T)) # convert es to Pa
end

# supply TK [Kelvin] by keyword, ignores positional T!!
function es(T,p=1e5; TK=T+C, P=p*1e-2)
    T = TK - C
    esat = 1e2 * 6.1121*(1.0007 + 3.46e-8*P)*exp((17.502*T)/(240.97 + T)) # convert es to Pa
end

"""
qs(p,T) is saturation specific humidity based on Wexler's formula for es
with enhancement factor (see es.m).
p [Pa], T [degrees C], qs [kg/kg]
From A. L. Buck 1981: JAM, 20, 1527-1532.
SPdeS 7 July 2004
"""
qs(p,T) = RdoRv*es(T,p) / (p + (RdoRv-1)*es(T,p))

"dqsdT(p,T[C]) derivative of qs with respect to T at p,T by autodiff of Bolton's qs"
dqsdT(p,T) = ForwardDiff.derivative(t -> qs(p,t), T)
# assumes isobaric

# wet bulb temperature methods
# for approximating the evap process

"General single Newton iteration to update x toward f(x) = fhat for a univariate function f"
updatex(f, x, fhat=0) = x + (fhat-f(x)) / ForwardDiff.derivative(f, x)

"""
Twet_autodiff(T[K], q[kg/kg], p[Pa]; niter=2) wet bulb temperature using Newton's method
for target specific humidity q[kg/kg]. Uses automatic differntiation.
"""
function Twet_autodiff(T, q, p; niter=2)
    f(t) = (t - T) + LvK((T+t)/2)/Cp * (qs(p,t-C) - q)
    t=T
    for i in 1:niter
        t = updatex(f, t, 0)
    end
    t
end
# 2 iterations converges to ~0.001 K

# call as...
# q = rh*qs(pa, Ta)
# Twet_autodiff(Ta, rh*qs(pa, Ta-C), pa) 

end # module VaporSat

using .VaporSat

ArgumentError: ArgumentError: Package ForwardDiff not found in current path.
- Run `import Pkg; Pkg.add("ForwardDiff")` to install the ForwardDiff package.

In [2]:
function extract_datetime(filename::String)
    m = match(r"(\d{8})_(\d{4})", filename)
    if m !== nothing
        date_str = m.captures[1]  # "20250704"
        time_str = m.captures[2]  # "0000"
        return DateTime(date_str * time_str, "yyyymmddHHMM")
    else
        error("No valid datetime found in filename.")
    end
end

extract_datetime (generic function with 1 method)

In [ ]:
amini_dir = "../data/uwyo/aminidivi/"
amini_files = filter(x -> startswith(x,"aminidivi") && endswith(x,".nc"), readdir(amini_dir))
dts = extract_datetime.(amini_files)

for f in amini_files
    dt = extract_datetime(f)
    println("Processing file: $f with datetime: $dt")
    ds = NCDatasets.Dataset(joinpath(amini_dir, f))
    
    # Extract variables
    pressure = ds["pressure"][:]
    temperature = ds["temperature"][:]
    dewpoint = ds["dewpoint"][:]
    wind_speed = ds["wind_speed"][:]
    wind_direction = ds["wind_direction"][:]
 
end

435-element Vector{DateTime}:
 2019-03-01T00:00:00
 2019-03-02T00:00:00
 2019-03-03T00:00:00
 2019-03-04T00:00:00
 2019-03-05T00:00:00
 2019-03-06T00:00:00
 2019-03-07T00:00:00
 2019-03-08T00:00:00
 2019-03-09T00:00:00
 2019-03-10T00:00:00
 ⋮
 2025-07-04T12:00:00
 2025-07-05T00:00:00
 2025-07-05T12:00:00
 2025-07-06T00:00:00
 2025-07-06T12:00:00
 2025-07-07T00:00:00
 2025-07-07T12:00:00
 2025-07-08T00:00:00
 2025-07-09T00:00:00

In [ ]:

    
    # Plotting
    fig, ax1 = subplots()
    ax1.plot(temperature, pressure, label="Temperature (K)", color="red")
    ax1.plot(dewpoint, pressure, label="Dewpoint (K)", color="blue")
    
    ax2 = ax1.twiny()
    ax2.plot(wind_speed, pressure, label="Wind Speed (m/s)", color="green")
    
    ax1.set_xlabel("Temperature / Dewpoint (K)")
    ax2.set_xlabel("Wind Speed (m/s)")
    ax1.set_ylabel("Pressure (hPa)")
    
    ax1.invert_yaxis()  # Invert y-axis for atmospheric profile
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    
    title_str = "Sounding at Aminidivi on $(dt)"
    plt.title(title_str)
    
    savefig(joinpath(amini_dir, "aminidivi_sounding_$(dt).png"))